# 01 – Inspect Raw Data
**Purpose:** Find every data-quality issue in the raw HDB data **before** cleaning (CRISP-DM: Data Understanding).
**Input:** `data/raw/hdb_resale_raw.csv`, `data/raw/hdb_rpi_raw.csv` (from `00_get_data`)
**Output:** printed checks + `reports/inspection_findings.csv` – every issue with its **actual count**, generated from the checks below (nothing typed by hand)
**Issues:** #7 Validate types and ranges · #8 Missing values and duplicates (discovery) – owner **Jaydon**

The ranges and the duplicate definition come from the same shared cell as `02_clean`, so both notebooks count the same way.

In [1]:
# Imports and paths
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
ROOT = Path('..').resolve()                        # repo root (notebook lives in notebooks/)
RAW_DIR = ROOT / 'data' / 'raw'
CLEAN_DIR = ROOT / 'data' / 'clean'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Shared rules - identical in 01_inspect and 02_clean so both report the same counts
RANGES = {'resale_price': (50_000, 2_500_000),               # S$
          'floor_area_sqm': (20, 300),                       # sqm
          'lease_commence_date': (1960, pd.Timestamp.today().year)}
DUP_COLS = ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm',
            'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']   # one sale = these fields, not _id

def dup_key(frame):
    """Comparable version of DUP_COLS: text, trimmed, single spaces, upper case, '67.0' -> '67'."""
    return frame[DUP_COLS].astype(str).apply(
        lambda s: s.str.strip().str.replace(r'\s+', ' ', regex=True).str.upper().str.replace(r'\.0$', '', regex=True))

In [2]:
# Load raw data as TEXT, exactly as the API delivered it; start the findings list
resale = pd.read_csv(RAW_DIR / 'hdb_resale_raw.csv', dtype=str)
rpi = pd.read_csv(RAW_DIR / 'hdb_rpi_raw.csv', dtype=str)
N = len(resale)
print(f'Resale: {resale.shape} | RPI: {rpi.shape}')

findings = []
def issue(column, problem, count, fix):
    """Record an issue only if it affects at least one row."""
    if count > 0:
        findings.append({'column': column, 'issue': problem, 'rows': int(count),
                         'pct': round(count / N * 100, 2), 'planned_fix_in_02_clean': fix})
resale.head()

Resale: (245351, 12) | RPI: (147, 3)


,_id,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,1,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44,Improved,1979,61 years 04 months,232000
1,2,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67,New Generation,1978,60 years 07 months,250000
2,3,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67,New Generation,1980,62 years 05 months,262000
3,4,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68,New Generation,1980,62 years 01 month,265000
4,5,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67,New Generation,1980,62 years 05 months,265000


In [3]:
# Column names and non-null counts
resale.info()

<class 'pandas.DataFrame'>
RangeIndex: 245351 entries, 0 to 245350
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   _id                  245351 non-null  str  
 1   month                245351 non-null  str  
 2   town                 245351 non-null  str  
 3   flat_type            245351 non-null  str  
 4   block                245351 non-null  str  
 5   street_name          245351 non-null  str  
 6   storey_range         245351 non-null  str  
 7   floor_area_sqm       245351 non-null  str  
 8   flat_model           245351 non-null  str  
 9   lease_commence_date  245351 non-null  str  
 10  remaining_lease      245351 non-null  str  
 11  resale_price         245351 non-null  str  
dtypes: str(12)
memory usage: 22.5 MB


## 1. Missing values

In [4]:
# Missing (NaN) and blank strings per column
miss = pd.DataFrame({'missing': resale.isna().sum(),
                     'blank': resale.apply(lambda s: s.str.strip().eq('').sum())})
miss['total'] = miss['missing'] + miss['blank']
for col, n in miss['total'].items():
    fix = 'Drop row (cannot place sale)' if col in ['month', 'town', 'resale_price'] else 'Keep row; impute or leave blank'
    issue(col, 'Missing or blank', n, fix)
miss[miss['total'] > 0] if miss['total'].any() else print('No missing or blank values')

No missing or blank values


## 2. Duplicates (same definition as 02_clean)

In [5]:
# _id must be unique; a duplicate sale = identical DUP_COLS after trimming and upper-casing
key = dup_key(resale)
dup_exact = resale.duplicated(subset=DUP_COLS, keep=False).sum()
dup_norm = key.duplicated(keep=False).sum()
print('Duplicate _id                               :', resale['_id'].duplicated().sum())
print('Rows in identical groups (raw text)         :', dup_exact)
print('Rows in identical groups (after normalising):', dup_norm)
issue('_id', 'Duplicate record ID', resale['_id'].duplicated().sum(), 'Stop - re-run 00_get_data')
issue('DUP_COLS', 'Identical sale records (same details, different _id)', dup_norm, 'Flag, keep (can be separate real sales)')
resale[key.duplicated(keep=False)].sort_values(DUP_COLS).head(10)

Duplicate _id                               : 0
Rows in identical groups (raw text)         : 635
Rows in identical groups (after normalising): 635


,_id,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
224,225,2017-01,BUKIT MERAH,4 ROOM,106,HENDERSON CRES,07 TO 09,81,Improved,1975,57 years,470000
243,244,2017-01,BUKIT MERAH,4 ROOM,106,HENDERSON CRES,07 TO 09,81,Improved,1975,57 years,470000
304,305,2017-01,CENTRAL AREA,3 ROOM,271,QUEEN ST,16 TO 18,68,Improved,1979,61 years 02 months,470000
305,306,2017-01,CENTRAL AREA,3 ROOM,271,QUEEN ST,16 TO 18,68,Improved,1979,61 years 02 months,470000
505,506,2017-01,JURONG EAST,4 ROOM,265,TOH GUAN RD,04 TO 06,101,Model A,1998,80 years 09 months,470000
510,511,2017-01,JURONG EAST,4 ROOM,265,TOH GUAN RD,04 TO 06,101,Model A,1998,80 years 09 months,470000
533,534,2017-01,JURONG WEST,4 ROOM,337A,TAH CHING RD,19 TO 21,92,Model A,2010,92 years 03 months,470000
591,592,2017-01,JURONG WEST,4 ROOM,337A,TAH CHING RD,19 TO 21,92,Model A,2010,92 years 03 months,470000
671,672,2017-01,PASIR RIS,4 ROOM,753,PASIR RIS ST 71,01 TO 03,105,Model A,1996,78 years 10 months,368000
672,673,2017-01,PASIR RIS,4 ROOM,753,PASIR RIS ST 71,01 TO 03,105,Model A,1996,78 years 10 months,368000


## 3. Types and value ranges

In [6]:
# Numeric fields: delivered as text; count values that fail to convert and values out of range
num = resale[list(RANGES)].apply(lambda s: pd.to_numeric(s.str.strip(), errors='coerce'))
issue('resale_price, floor_area_sqm, lease_commence_date', 'Numbers delivered as text', N, 'pd.to_numeric')
for col, (lo, hi) in RANGES.items():
    bad_type = (num[col].isna() & resale[col].notna()).sum()
    out = (~num[col].between(lo, hi) & num[col].notna()).sum()
    print(f'{col:20s} not numeric: {bad_type:,}  outside {lo:,}-{hi:,}: {out:,}')
    issue(col, 'Not numeric', bad_type, 'Set to NaN, count')
    issue(col, f'Outside expected range {lo:,}-{hi:,}', out, 'Flag, keep for review')
num.describe().round(1)

resale_price         not numeric: 0  outside 50,000-2,500,000: 0
floor_area_sqm       not numeric: 0  outside 20-300: 1
lease_commence_date  not numeric: 0  outside 1,960-2,026: 0


,resale_price,floor_area_sqm,lease_commence_date
count,245351.0,245351.0,245351.0
mean,538050.3,96.7,1996.7
std,194593.0,24.0,14.4
min,140000.0,31.0,1966.0
25%,394000.0,81.0,1985.0
50%,507500.0,93.0,1997.0
75%,645888.0,112.0,2012.0
max,1818000.0,366.7,2023.0


## 4. Dates

In [7]:
# month should be YYYY-MM text -> needs converting to a real date
ok = resale['month'].str.strip().str.fullmatch(r'\d{4}-\d{2}')
print('Range:', resale['month'].min(), 'to', resale['month'].max(), '| not YYYY-MM:', (~ok.fillna(False)).sum())
issue('month', 'Text YYYY-MM, not a date', N, 'Convert to ISO 8601 date (first of month) + quarter')
issue('month', 'Not in YYYY-MM format', (~ok.fillna(False)).sum(), 'Drop row, count')
resale['month'].value_counts().sort_index().tail(12)                   # last 12 months of volume

Range: 2017-01 to 2026-11 | not YYYY-MM: 0


month
2025-12    2024
2026-01    2338
2026-02    1661
2026-03    2042
2026-04    1933
2026-05    2126
2026-06    2124
2026-07    2647
2026-08    2518
2026-09    2239
2026-10    2173
2026-11    1484
Name: count, dtype: int64

## 5. Text fields – categories, casing and formats

In [8]:
# Same value written differently (case / spaces)? Count values and rows affected
for col in ['town', 'flat_type', 'flat_model', 'street_name']:
    clean = resale[col].str.strip().str.replace(r'\s+', ' ', regex=True)
    key_ = clean.str.upper()
    canonical = clean.groupby(key_).agg(lambda s: s.value_counts().idxmax())   # most common spelling per value
    differs = (clean != key_.map(canonical)).sum() + (resale[col] != clean).sum()
    print(f'{col:12s} unique raw: {resale[col].nunique():4d}  after normalising: {key_.nunique():4d}  rows to fix: {differs:,}')
    issue(col, 'Same value in different case/spacing', differs, 'Trim; keep most common official spelling')
resale['flat_model'].value_counts().head(25)

town         unique raw:   26  after normalising:   26  rows to fix: 0
flat_type    unique raw:    7  after normalising:    7  rows to fix: 0
flat_model   unique raw:   21  after normalising:   21  rows to fix: 0
street_name  unique raw:  580  after normalising:  580  rows to fix: 0


flat_model
Model A                   88502
Improved                  59994
New Generation            29671
Premium Apartment         26748
Simplified                 9335
Apartment                  8765
Maisonette                 6712
Standard                   6511
DBSS                       3888
Model A2                   2734
2-room                      542
Model A-Maisonette          426
Adjoined flat               419
Type S1                     400
Type S2                     201
Premium Apartment Loft      143
Terrace                     123
Multi Generation             94
3Gen                         88
Improved-Maisonette          33
Premium Maisonette           22
Name: count, dtype: int64

In [9]:
# remaining_lease, e.g. '61 years 04 months' or '70 years' -> needs parsing to a number
pattern = resale['remaining_lease'].str.replace(r'\d+', 'N', regex=True)
print(pattern.value_counts().to_string())
fmt_ok = resale['remaining_lease'].str.fullmatch(r'\d+ years?( \d+ months?)?').fillna(False)
issue('remaining_lease', 'Text such as 61 years 04 months', fmt_ok.sum(), 'Parse to years (float)')
issue('remaining_lease', 'Unexpected format', (~fmt_ok & resale['remaining_lease'].notna()).sum(), 'Compute from lease start, flag')

remaining_lease
N years N months    205386
N years              20118
N years N month      19847


In [10]:
# storey_range, e.g. '10 TO 12' -> needs splitting into numbers
st_ok = resale['storey_range'].str.fullmatch(r'\d{2} TO \d{2}').fillna(False)
print(st_ok.value_counts())
issue('storey_range', 'Text range such as 10 TO 12', st_ok.sum(), 'Split to low / high / mid floor')
issue('storey_range', 'Unexpected format', (~st_ok & resale['storey_range'].notna()).sum(), 'Leave NaN, flag')

storey_range
True    245351
Name: count, dtype: int64


## 6. Resale Price Index

In [11]:
# Quarter format YYYY-Qn, numeric index, blank rows, coverage vs the transactions
q_ok = rpi['quarter'].notna()
last_q = rpi.loc[q_ok, 'quarter'].max()
last_q_end = pd.Period(last_q.replace('-', ''), freq='Q').end_time
after = (pd.to_datetime(resale['month'], format='%Y-%m', errors='coerce') > last_q_end).sum()
print('Blank quarter rows:', (~q_ok).sum(), '| range:', rpi.loc[q_ok, 'quarter'].min(), 'to', last_q)
print('Transactions after the last published index quarter:', after)
issue('rpi.quarter', 'Blank quarter row in the index', (~q_ok).sum(), 'Drop row')
issue('rpi.index', 'Not numeric', (pd.to_numeric(rpi['index'], errors='coerce').isna() & q_ok).sum(), 'Set to NaN')
issue('rpi coverage', f'Transactions after last index quarter ({last_q})', after, 'Keep; rpi_index blank, report')
rpi.tail()

Blank quarter rows: 0 | range: 1990-Q1 to 2026-Q3
Transactions after the last published index quarter: 3657


,_id,quarter,index
142,143,2025-Q3,203.7
143,144,2025-Q4,203.6
144,145,2026-Q1,203.4
145,146,2026-Q2,202.8
146,147,2026-Q3,202.5


## Inspection findings (generated from the checks above)

In [12]:
# Every issue with its real count -> saved as evidence for the checklist and 02_clean
report = pd.DataFrame(findings)
(ROOT / 'reports').mkdir(exist_ok=True)
report.to_csv(ROOT / 'reports' / 'inspection_findings.csv', index=False)
pd.set_option('display.max_colwidth', None)
report

,column,issue,rows,pct,planned_fix_in_02_clean
0,DUP_COLS,"Identical sale records (same details, different _id)",635,0.26,"Flag, keep (can be separate real sales)"
1,"resale_price, floor_area_sqm, lease_commence_date",Numbers delivered as text,245351,100.00,pd.to_numeric
2,floor_area_sqm,Outside expected range 20-300,1,0.00,"Flag, keep for review"
3,month,"Text YYYY-MM, not a date",245351,100.00,Convert to ISO 8601 date (first of month) + quarter
4,remaining_lease,Text such as 61 years 04 months,245351,100.00,Parse to years (float)
5,storey_range,Text range such as 10 TO 12,245351,100.00,Split to low / high / mid floor
6,rpi coverage,Transactions after last index quarter (2026-Q3),3657,1.49,"Keep; rpi_index blank, report"
